## 01 — Univariate EDA

> **Univariate exploratory analysis of the cleaned NBA dataset.**  
> Objective: understand each variable in isolation, identify anomalous patterns, quantify target balance, and establish descriptive baselines.  
> **Input:** `data/2_processed/cleaned_nba_data_2000-01_2025-26.csv`  
> **Output:** descriptive statistics, distribution plots, temporal coverage checks, and categorical balance summaries.

---

**Pipeline**

1. Setup — imports, configuration, plotting defaults, and variable groups
2. Data Loading — versioned cleaned dataset retrieval and schema confirmation
3. Numerical Statistics Overview — central tendency, dispersion, and range diagnostics
4. Performance Variable Distributions — histograms, KDE plots, and boxplots
5. Temporal Variable Distributions — season, month, and team-season coverage checks
6. Categorical Variables — win/loss balance and home/away balance

### 1. Setup

* **What:** Imports statistical, visualization, and table-rendering libraries, loads project configuration, and defines plotting constants for NBA-themed figures.
* **Why:** To create a reproducible exploratory analysis environment before computing descriptive statistics or rendering plots.
* **Reasoning/Choices:** Consistent display settings and fixed variable groups make the EDA easier to compare across seasons, teams, and later bivariate analysis notebooks.

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from scipy import stats

import itables.options as opt
from itables import init_notebook_mode, show

from src.loader import load_config

warnings.filterwarnings("ignore")

config = load_config()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False
opt.maxBytes   = 0
opt.pageLength = 15
opt.lengthMenu = [15, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# ── Plotting configuration ──────────────────────────────────────────────────
NBA_PALETTE   = ["#1d428a", "#c8102e", "#fdb927", "#007a33", "#552583", "#ce1141"]
ACCENT        = "#1d428a"
NEUTRAL       = "#6c757d"
FIG_W_SINGLE  = 10
FIG_H_SINGLE  = 4
FIG_W_GRID    = 16
FIG_H_ROW     = 4
DPI           = 130

sns.set_theme(style="whitegrid", palette=NBA_PALETTE)
plt.rcParams.update({
    "figure.dpi"      : DPI,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "font.size"       : 11,
    "axes.titlesize"  : 13,
    "axes.labelsize"  : 11,
})

NOTEBOOK_VERSION = "1.0.0"
log.info(f"EDA Univariata v{NOTEBOOK_VERSION} — initialized")

#### 1.1 Data Loading

* **What:** Resolves the cleaned dataset path from configuration and loads the versioned processed CSV with stable identifier types.
* **Why:** To ensure the analysis uses the same cleaned artifact produced by the data-cleaning notebook.
* **Reasoning/Choices:** Parsing dates while preserving game and team identifiers avoids accidental type drift and keeps temporal analysis reliable.

In [ ]:
start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"
DATASET_VERSION = f"{start_season}_{end_season}"

PROCESSED_DIR = ROOT / config["data"]["processed_data_dir"]
CLEAN_FILE    = PROCESSED_DIR / f"cleaned_nba_data_{DATASET_VERSION}.csv"

df = pl.read_csv(
    CLEAN_FILE,
    try_parse_dates=True,
    schema_overrides={"game_id": pl.Int64, "team_id": pl.Int64},
)

log.info(f"Loaded  : {CLEAN_FILE.name}")
log.info(f"Shape   : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
log.info(f"Seasons : {df['season'].n_unique()}  ({df['season'].min()} – {df['season'].max()})")
log.info(f"Teams   : {df['team_abbreviation'].n_unique()}")

#### 1.2 Variable List Definition

* **What:** Groups numerical, categorical, and performance variables used throughout the univariate analysis.
* **Why:** To separate basketball performance measures, temporal fields, and target labels before computing distributions.
* **Reasoning/Choices:** Domain-driven thresholds for points help flag unusual scoring games without automatically treating valid NBA outliers as data errors.

In [ ]:
# Numerical variables — grouped by domain
PERF_VARS = ["pts", "off_rating", "def_rating", "net_rating", "pace", "ts_pct"]

# Sanity check: plus_minus excluded in feature engineering
ALL_NUM_VARS = ["pts", "plus_minus", "off_rating", "def_rating", "net_rating",
                "pace", "ts_pct", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta"]

# Categorical variables
CAT_VARS = ["wl", "home_away", "season"]

# Domain-driven outlier thresholds for PTS (NBA knowledge)
PTS_LOW  = 70
PTS_HIGH = 160

log.info(f"Numerical Variables  : {len(ALL_NUM_VARS)} — {ALL_NUM_VARS}")
log.info(f"Categorical Variables: {len(CAT_VARS)} — {CAT_VARS}")

# Dataset schema preview
show(
    pl.DataFrame({
        "column"  : df.columns,
        "dtype"    : [str(t) for t in df.dtypes],
        "null_cnt" : [df[c].null_count() for c in df.columns],
        "domain"  : [
            "id" if c in ("game_id", "team_id") else
            "date" if c == "game_date" else
            "time-based" if c == "season" else
            "categorical" if c in CAT_VARS else
            "numerical" if c in ALL_NUM_VARS else
            "flag"
            for c in df.columns
        ],
    }).to_pandas(),
    caption="Schema of the cleaned dataset",
)

### 2. Numerical Statistics Overview

* **What:** Computes descriptive statistics for each numerical variable, including central tendency, spread, and quartiles.
* **Why:** To establish baseline distributions before inspecting plots or comparing variables against the win/loss target.
* **Reasoning/Choices:** Mean, median, standard deviation, and interquartile range together reveal skewness, volatility, and potential outliers more clearly than a single summary measure.

In [ ]:
# Compute extended descriptive statistics using scipy
rows = []
for col in ALL_NUM_VARS:
    series = df[col].drop_nulls().to_numpy().astype(float)
    rows.append({
        "variable" : col,
        "count"    : int(len(series)),
        "mean"     : round(float(np.mean(series)), 4),
        "median"   : round(float(np.median(series)), 4),
        "std"      : round(float(np.std(series, ddof=1)), 4),
        "min"      : round(float(np.min(series)), 4),
        "max"      : round(float(np.max(series)), 4),
        "q25"      : round(float(np.percentile(series, 25)), 4),
        "q75"      : round(float(np.percentile(series, 75)), 4),
    })

desc_df = pl.DataFrame(rows)

show(
    desc_df.to_pandas(),
    caption="Extended descriptive statistics for numerical variables",
)

### 3. Performance Variable Distributions

* **What:** Visualizes the marginal distributions of core basketball performance metrics such as points, ratings, pace, and true shooting percentage.
* **Why:** To understand how each modeling signal behaves independently before building multivariate relationships.
* **Reasoning/Choices:** Distribution shape matters for feature engineering: skewed variables, heavy tails, or bounded percentages may need transformations, scaling, or careful interpretation.

#### 3.1 Histogram and KDE

* **What:** Combines histograms with kernel density estimates for each performance variable.
* **Why:** To compare empirical frequency with smoothed distribution shape in a compact visual form.
* **Reasoning/Choices:** KDE overlays help reveal modality and skew, while histograms preserve the actual count structure of NBA team-game observations.

In [ ]:
# Calculate the number of rows needed based on the number of variables and columns
n_cols = 3
n_rows = int(np.ceil(len(PERF_VARS) / n_cols))

# Create the figure and subplots
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(FIG_W_GRID, FIG_H_ROW * n_rows))
axes = axes.flatten()

# Iterate through each performance variable to plot its distribution
for i, col in enumerate(PERF_VARS):
    ax   = axes[i]
    # Extract data and clean null values
    data = df[col].drop_nulls().to_numpy().astype(float)
    # Retrieve descriptive statistics for the current variable
    row_info = desc_df.filter(pl.col("variable") == col).row(0, named=True)

    # Plot histogram with Kernel Density Estimation (KDE)
    sns.histplot(data, bins=50, kde=True, color=ACCENT,
                 edgecolor="white", linewidth=0.3, ax=ax)

    # Add vertical lines for mean and median
    ax.axvline(row_info["mean"],   color="#c8102e", linewidth=1.5,
               linestyle="--", label=f"mean={row_info['mean']:.1f}")
    ax.axvline(row_info["median"], color="#fdb927", linewidth=1.5,
               linestyle=":",  label=f"med={row_info['median']:.1f}")

    # Highlight specific anomaly thresholds for 'pts' variable
    if col == "pts":
        ax.axvline(PTS_LOW,  color="#552583", linewidth=1.2, linestyle="-.",
                   label=f"PTS<{PTS_LOW}")
        ax.axvline(PTS_HIGH, color="#552583", linewidth=1.2, linestyle="-.",
                   label=f"PTS>{PTS_HIGH}")

    # Set chart labels and titles
    ax.set_title(f"{col.upper()} Distribution")
    ax.set_xlabel(col.upper())
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8, loc="upper right")

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Add main title to the figure
fig.suptitle("Performance Variable Distributions — Histogram + KDE",
             fontsize=14, y=1.01)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

#### 3.2 Performance Variable Boxplots

* **What:** Uses boxplots to summarize median, interquartile range, and extreme observations for key performance variables.
* **Why:** To identify dispersion and outlier structure without assuming a normal distribution.
* **Reasoning/Choices:** In NBA data, extreme values can be legitimate high-scoring or low-efficiency games, so boxplots are used for investigation rather than automatic removal.

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(FIG_W_GRID, FIG_H_ROW * n_rows))
axes = axes.flatten()

for i, col in enumerate(PERF_VARS):
    ax   = axes[i]
    # Extract numerical data and remove null values for plotting
    data = df[col].drop_nulls().to_numpy().astype(float)

    # Create boxplot with custom styling
    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=ACCENT, alpha=0.7),
                    medianprops=dict(color="#fdb927", linewidth=2),
                    flierprops=dict(marker=".", color="#c8102e",
                                    markersize=3, alpha=0.4),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2))

    # Add horizontal lines for threshold anomalies if the column is 'pts'
    if col == "pts":
        ax.axhline(PTS_LOW,  color="#552583", linewidth=1.2, linestyle="-.",
                    label=f"<{PTS_LOW}")
        ax.axhline(PTS_HIGH, color="#552583", linewidth=1.2, linestyle="-.",
                    label=f">{PTS_HIGH}")
        ax.legend(fontsize=8)

    # Set plot labels and titles
    ax.set_title(col.upper())
    ax.set_ylabel(col.upper())
    ax.set_xticks([]) # Hide x-axis ticks for cleaner layout

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Add main figure title
fig.suptitle("Performance Variable Boxplots", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Count and print PTS anomalies
pts_low_cnt  = df.filter(pl.col("pts") < PTS_LOW).shape[0]
pts_high_cnt = df.filter(pl.col("pts") > PTS_HIGH).shape[0]
print(f"Games with PTS < {PTS_LOW}  : {pts_low_cnt}")
print(f"Games with PTS > {PTS_HIGH} : {pts_high_cnt}")

### 4. Temporal Variable Distributions

* **What:** Examines how games are distributed across seasons, months, and team-season coverage.
* **Why:** To detect calendar imbalance, missing seasons, shortened schedules, or era-specific effects that could influence model training.
* **Reasoning/Choices:** Temporal structure is central in sports prediction because rule changes, pandemic seasons, and schedule density can shift feature distributions over time.

#### 4.1 Games per Season

* **What:** Counts team-game records by NBA season.
* **Why:** To verify coverage and identify seasons with abnormal volume.
* **Reasoning/Choices:** Lower counts may be expected for lockout or pandemic-affected seasons, so interpretation should combine the plot with NBA schedule context.

In [ ]:
# Group data by season and count the rows
games_per_season = (
    df.group_by("season")
      .agg(pl.len().alias("n_game_rows"))
      .sort("season")
)

# Each season has 2 rows per game (home + away) → divide by 2 for unique games
games_per_season = games_per_season.with_columns(
    (pl.col("n_game_rows") // 2).alias("n_games")
)

# Prepare lists for plotting
seasons = games_per_season["season"].to_list()
counts  = games_per_season["n_games"].to_list()
mean_games = np.mean(counts)

# Create the figure and bar chart
fig, ax = plt.subplots(figsize=(FIG_W_GRID, FIG_H_SINGLE + 1))
# Define bar colors: highlight outliers (±15% from mean) in red, others in ACCENT color
bar_colors = [
    "#c8102e" if abs(c - mean_games) > 0.15 * mean_games else ACCENT
    for c in counts
]
ax.bar(seasons, counts, color=bar_colors, edgecolor="white", linewidth=0.4)
# Add a horizontal line representing the average number of games
ax.axhline(mean_games, color="#fdb927", linewidth=1.5, linestyle="--",
           label=f"mean={mean_games:.0f}")
ax.set_xlabel("Season")
ax.set_ylabel("Number of games")
ax.set_title("Distribution of games per season (unique games)")
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

# Print identified outliers
print(f"Season outliers (>±15% from mean {mean_games:.0f}):")
for s, c in zip(seasons, counts):
    if abs(c - mean_games) > 0.15 * mean_games:
        print(f"  {s}: {c} games  ({(c - mean_games) / mean_games:+.1%} from mean)")

#### 4.2 Games Distribution per Month

* **What:** Aggregates game counts by calendar month.
* **Why:** To expose seasonality in the regular-season schedule and confirm that the dataset follows expected NBA calendar patterns.
* **Reasoning/Choices:** Month-level counts help explain temporal density, rest patterns, and possible model sensitivity to compressed schedule periods.

In [ ]:
# Map month numbers to Italian abbreviations (translated to English)
MONTH_LABELS = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

# Extract the month from the 'game_date' column
df_month = df.with_columns(
    pl.col("game_date").dt.month().alias("month")
)

# Aggregate unique games (dividing by 2 because of home/away rows) by month
games_per_month = (
    df_month.group_by("month")
            .agg((pl.len() // 2).alias("n_games"))
            .sort("month")
)

# Prepare lists for plotting
months = games_per_month["month"].to_list()
month_labels = [MONTH_LABELS[m] for m in months]
month_counts = games_per_month["n_games"].to_list()

# Create the bar chart for seasonal distribution
fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_H_SINGLE))
ax.bar(month_labels, month_counts, color=ACCENT, edgecolor="white", linewidth=0.4)
ax.set_xlabel("Month")
ax.set_ylabel("Number of games")
ax.set_title("Intra-seasonal seasonality: Games per Month")
plt.tight_layout()
plt.show()

# Print explanatory note regarding the NBA schedule
print("Note: The NBA season runs from October/November through June (including playoffs).")

#### 4.3 Games per Team per Season — Coverage Check

* **What:** Measures how many observations each franchise contributes within each season.
* **Why:** To detect team-level coverage gaps that would bias rolling features, win-rate estimates, or train/test splits.
* **Reasoning/Choices:** Team-season completeness is more informative than global row count because modeling features are later built from each team's chronological history.

In [ ]:
# Group data by season and team to count total rows
games_team_season = (
    df.group_by(["season", "team_abbreviation"])
      .agg(pl.len().alias("n_rows"))
      .sort(["season", "team_abbreviation"])
)

# Pivot the data: teams as rows, seasons as columns
pivot = (
    games_team_season
    .pivot(on="season", index="team_abbreviation", values="n_rows", aggregate_function="sum")
    .sort("team_abbreviation")
)

# Convert to pandas for heatmap visualization and sort columns by season
pivot_pd = pivot.to_pandas().set_index("team_abbreviation")
pivot_pd = pivot_pd.reindex(sorted(pivot_pd.columns), axis=1)

# Generate heatmap to visualize data coverage per team and season
fig, ax = plt.subplots(figsize=(FIG_W_GRID + 6, max(8, len(pivot_pd) * 0.35)))
sns.heatmap(
    pivot_pd,
    ax=ax,
    cmap="Blues",
    linewidths=0.3,
    linecolor="#e0e0e0",
    annot=False,
    cbar_kws={"label": "Number of rows (home+away)"},
    mask=pivot_pd.isna(),
)
ax.set_title("Rows per team per season (dataset coverage)", fontsize=13)
ax.set_xlabel("Season")
ax.set_ylabel("Team")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Identify teams with anomalous coverage (less than 50% of the seasonal median)
season_min = games_team_season.group_by("season").agg(pl.col("n_rows").median().alias("median_rows"))
anomalous = (
    games_team_season
    .join(season_min, on="season")
    .filter((pl.col("n_rows") < pl.col("median_rows") * 0.5))
    .sort(["season", "n_rows"])
)

# Output summary of identified anomalies
print(f"\nTeams with coverage < 50% of the seasonal median: {anomalous.shape[0]}")
if anomalous.shape[0] > 0:
    show(anomalous.to_pandas(), caption="Teams with anomalous coverage")

### 5. Categorical Variables

* **What:** Reviews categorical distributions for outcome labels, venue labels, and season identifiers.
* **Why:** To evaluate class balance and confirm that key categorical fields are suitable for downstream modeling.
* **Reasoning/Choices:** Binary target balance and home/away symmetry define the baseline difficulty of NBA win prediction and guide metric interpretation.

#### 5.1 W/L Balance

* **What:** Counts wins and losses in the cleaned team-game dataset.
* **Why:** To quantify target balance before using accuracy, F1 score, or ROC-AUC in model evaluation.
* **Reasoning/Choices:** Because every NBA game produces one win and one loss at team level, large imbalance would signal a data issue or a filtering side effect.

In [ ]:
# Calculate counts for Wins (W) and Losses (L)
wl_counts = (
    df.group_by("wl")
      .agg(pl.len().alias("n"))
      .sort("wl")
)

# Calculate the percentage for each category
total = wl_counts["n"].sum()
wl_counts = wl_counts.with_columns(
    (pl.col("n") / total * 100).round(2).alias("pct")
)

# Print the distribution summary
print("W/L Distribution:")
for row in wl_counts.iter_rows(named=True):
    print(f"  {row['wl']}: {row['n']:,}  ({row['pct']:.2f}%)")

# Create a figure with two subplots: bar chart and pie chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors_wl = ["#1d428a", "#c8102e"]
axes[0].bar(wl_counts["wl"].to_list(), wl_counts["n"].to_list(),
            color=colors_wl, edgecolor="white")
# Add a reference line at 50%
axes[0].axhline(total / 2, color=NEUTRAL, linestyle="--", linewidth=1,
                label="50% line")
axes[0].set_title("W/L Count")
axes[0].set_ylabel("Number of rows")
axes[0].legend(fontsize=9)

# Pie chart
axes[1].pie(
    wl_counts["n"].to_list(),
    labels=[f"{r['wl']} ({r['pct']:.1f}%)" for r in wl_counts.iter_rows(named=True)],
    colors=colors_wl,
    startangle=90,
    wedgeprops={"edgecolor": "white"},
)
axes[1].set_title("W/L Proportion")

# Set the main figure title
fig.suptitle("W/L Target Balance (each game has 1 W + 1 L → 50/50 expected)",
             fontsize=12)
plt.tight_layout()
plt.show()

#### 5.2 Home / Away Balance

* **What:** Compares the number of home and away observations.
* **Why:** To verify that each game contributes one home team and one away team record.
* **Reasoning/Choices:** Balanced venue labels are required before estimating home-court advantage or constructing home-versus-away feature differentials.

In [ ]:
# Calculate counts for Home and Away games
ha_counts = (
    df.group_by("home_away")
      .agg(pl.len().alias("n"))
      .sort("home_away")
)

# Calculate the percentage for each category
total_ha = ha_counts["n"].sum()
ha_counts = ha_counts.with_columns(
    (pl.col("n") / total_ha * 100).round(2).alias("pct")
)

# Print the distribution summary
print("Home/Away Distribution:")
for row in ha_counts.iter_rows(named=True):
    print(f"  {row['home_away']}: {row['n']:,}  ({row['pct']:.2f}%)")

# Create a figure with two subplots: bar chart and pie chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors_ha = ["#007a33", "#ce1141"]
axes[0].bar(ha_counts["home_away"].to_list(), ha_counts["n"].to_list(),
            color=colors_ha, edgecolor="white")
# Add a reference line at 50%
axes[0].axhline(total_ha / 2, color=NEUTRAL, linestyle="--", linewidth=1,
                label="50% line")
axes[0].set_title("Home/Away Count")
axes[0].set_ylabel("Number of rows")
axes[0].legend(fontsize=9)

# Pie chart
axes[1].pie(
    ha_counts["n"].to_list(),
    labels=[f"{r['home_away']} ({r['pct']:.1f}%)" for r in ha_counts.iter_rows(named=True)],
    colors=colors_ha,
    startangle=90,
    wedgeprops={"edgecolor": "white"},
)
axes[1].set_title("Home/Away Proportion")

# Set the main figure title
fig.suptitle("Home/Away Balance (expected exactly 50/50)", fontsize=12)
plt.tight_layout()
plt.show()